# Fine-tuning Multi-modelo — TAIL/Sefaz NF-e

Destila o pipeline `model_v2_1` (Gemini 2.5 Flash-Lite) em SLMs locais via QLoRA usando o `finetune_dataset_augmented.jsonl` (1400 pares — 200 originais × 7 variantes).

**Modelos suportados:**
- `unsloth/Phi-3.5-mini-instruct`
- `unsloth/Llama-3.2-3B-Instruct`
- `unsloth/Qwen2.5-3B-Instruct`

**Pipeline:**
1. Parse JSONL e split **por original** (150 train / 50 eval) — evita leakage entre variantes
2. Treina QLoRA por até 2 épocas com early stopping (~30-45min L4)
3. Roda inferência nos 350 records do holdout (50 originais × 7)
4. Computa **consistency benchmark** via `benchmark.py` + agreement com Gemini
5. Comparação: modelo zero-shot vs modelo fine-tuned vs Gemini (upper bound)

**Métricas reportadas:**
- *mean_constancy* / *weighted_constancy* (benchmark.py): consistência média entre as 7 variantes
- *Field agreement vs Gemini*: fração de campos que batem com o target
- *JSON validity rate*

## 0 — GPU sanity

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB


## 1 — Install

Unsloth: ~2x speedup, ~60% menos VRAM.

In [ ]:
%%capture
!pip install --upgrade -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps -q "trl>=0.21" peft accelerate bitsandbytes
!pip install -q datasets pandas pyarrow

## 2 — Config

In [ ]:
import os, re, json, random, sys
from collections import defaultdict, Counter
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from datasets import Dataset

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# ── TROQUE AQUI para cada modelo ─────────────────────────────
MODEL_NAME = "unsloth/Phi-3.5-mini-instruct"
# MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
# MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

MAX_SEQ_LENGTH  = 1536
LOAD_IN_4BIT    = True

# ── Chat tokens por modelo (automático) ──────────────────────
_CHAT_TOKENS = {
    "llama":  ("user", "<|start_header_id|>user<|end_header_id|>\n\n",
                       "<|start_header_id|>assistant<|end_header_id|>\n\n"),
    "phi":    ("user", "<|user|>\n",
                       "<|assistant|>\n"),
    "qwen":   ("user", "<|im_start|>user\n",
                       "<|im_start|>assistant\n"),
}

def get_chat_tokens(model_name: str):
    name = model_name.lower()
    if "llama" in name: return _CHAT_TOKENS["llama"][1], _CHAT_TOKENS["llama"][2]
    if "phi"   in name: return _CHAT_TOKENS["phi"][1],   _CHAT_TOKENS["phi"][2]
    return _CHAT_TOKENS["qwen"][1], _CHAT_TOKENS["qwen"][2]

INSTRUCTION_PART, RESPONSE_PART = get_chat_tokens(MODEL_NAME)

# ── LoRA ─────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

# ── Treino ───────────────────────────────────────────────────
_short = MODEL_NAME.split("/")[-1].lower().replace(".", "").replace("-instruct", "")
RUN_NAME   = f"{_short}-nfe-r{LORA_R}"
OUTPUT_DIR = f"outputs/{RUN_NAME}"
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR         = 1.5e-4

os.makedirs("data", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Run         : {RUN_NAME}")
print(f"Model       : {MODEL_NAME}")
print(f"Instr token : {repr(INSTRUCTION_PART)}")
print(f"Resp  token : {repr(RESPONSE_PART)}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
print(f"BF16        : {torch.cuda.is_bf16_supported()}")

Run         : phi-35-mini-nfe-r16
Model       : unsloth/Phi-3.5-mini-instruct
Instr token : '<|user|>\n'
Resp  token : '<|assistant|>\n'
GPU         : Tesla T4
VRAM total  : 15.6GB
BF16        : True


## 3 — Upload arquivos

Faça upload de:
1. **`finetune_dataset_augmented.jsonl`** (obrigatório) — dataset de treino
2. **`benchmark.py`** (obrigatório) — módulo de benchmark do repo

`resultado_pipeline_v2.csv` é opcional — só resolve nomes de originais reais.


In [ ]:
JSONL_PATH     = '/content/finetune_dataset_augmented.jsonl'
BENCHMARK_PATH = '/content/benchmark.py'
V2_CSV_PATH    = None

# Torna benchmark importável
sys.path.insert(0, ".")

## 4 — Parse JSONL + split BY ORIGINAL

**Crítico:** split por **target JSON** (= por original), não por record. Todas as 7 variantes de um original ficam do mesmo lado. Splitar por record vazaria informação entre attacks da mesma origem.

In [ ]:
records = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total records: {len(records)}")

# Validar estrutura
assert all("messages" in r and len(r["messages"]) == 3 for r in records)
assert all(r["messages"][0]["role"] == "system" for r in records)

parse_errors = 0
for r in records:
    try:
        ast = json.loads(r["messages"][2]["content"])
        assert isinstance(ast, list) and len(ast) == 1 and "tipo_produto" in ast[0]
    except Exception:
        parse_errors += 1
print(f"Assistant JSON errors: {parse_errors}/{len(records)}")

# ── Helpers de extração ─────────────────────────────────────
_TITLE_RE = re.compile(r'Título:\s*"(.*)"')

def get_title(rec):
    m = _TITLE_RE.search(rec["messages"][1]["content"])
    return m.group(1) if m else rec["messages"][1]["content"]

def target_key(rec):
    """Chave determinística do target JSON (ignora idx) — agrupa as 7 variantes."""
    ast = json.loads(rec["messages"][2]["content"])[0]
    payload = {k: v for k, v in ast.items() if k != "idx"}
    return json.dumps(payload, sort_keys=True, ensure_ascii=False)

def target_dict(rec):
    return json.loads(rec["messages"][2]["content"])[0]

# ── Resolver "titulo original" por grupo ────────────────────
# Se o CSV v2 foi upado, usa titulo_pre_norm como original; senão, ID sintético.
group_to_original = {}
if V2_CSV_PATH:
    df_v2 = pd.read_csv(V2_CSV_PATH)
    print(f"\nResolvendo originais via {V2_CSV_PATH} ({len(df_v2)} linhas)")
    for _, row in df_v2.iterrows():
        try:
            tj = json.loads(row["json_limpo_python"])
            payload = {k: v for k, v in tj.items() if k != "idx"}
            key = json.dumps(payload, sort_keys=True, ensure_ascii=False)
            group_to_original[key] = row["titulo_pre_norm"]
        except Exception:
            continue
    print(f"Mapeados {len(group_to_original)} targets → títulos originais reais")

# ── Group records by target ─────────────────────────────────
groups = defaultdict(list)
for r in records:
    groups[target_key(r)].append(r)

print(f"Grupos únicos: {len(groups)}")
print(f"Tamanhos de grupo: {dict(sorted(Counter(len(v) for v in groups.values()).items()))}")
print(f"Média variantes/grupo: {len(records)/len(groups):.2f}")

# Sintéticos pra grupos sem match (ou se V2 CSV não upado)
for i, key in enumerate(sorted(groups.keys())):
    if key not in group_to_original:
        group_to_original[key] = f"orig_{i:04d}"

# ── Split por original ──────────────────────────────────────
all_origs = sorted(groups.keys())
random.Random(RANDOM_SEED).shuffle(all_origs)
N_EVAL_ORIGS = 50
eval_origs  = set(all_origs[:N_EVAL_ORIGS])
train_origs = set(all_origs[N_EVAL_ORIGS:])

train_records = [r for k in train_origs for r in groups[k]]
eval_records  = [r for k in eval_origs  for r in groups[k]]

random.Random(RANDOM_SEED).shuffle(train_records)
# eval: ordenar por target_key pra deixar grupos contíguos (debug-friendly)
eval_records.sort(key=lambda r: target_key(r))

print(f"\nTrain: {len(train_records)} records ({len(train_origs)} originais)")
print(f"Eval:  {len(eval_records)} records ({len(eval_origs)} originais)")

Total records: 2353
Assistant JSON errors: 0/2353
Grupos únicos: 495
Tamanhos de grupo: {1: 7, 2: 36, 3: 71, 4: 86, 5: 121, 6: 106, 7: 68}
Média variantes/grupo: 4.75

Train: 2115 records (445 originais)
Eval:  238 records (50 originais)


## 5 — Reconstrução do `clean_llm_output` + `build_match_key`

Espelha o `eval.py` do pipeline v2.1 (regex do cell 22) — se você tem o `eval.py` original, suba e troque por `from eval import ...`. Por enquanto reconstrução inline pra não bloquear.

In [ ]:
_RE_SPACES    = re.compile(r"\s+")
_RE_VOLUME_ML = re.compile(r"(\d+(?:[.,]\d+)?)\s*ml", re.I)
_RE_VOLUME_L  = re.compile(r"(\d+(?:[.,]\d+)?)\s*l(?:itro[s]?)?(?=\b|$)", re.I)
_RE_WEIGHT_KG = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:g|ilos?|ilogramas?)", re.I)
_RE_WEIGHT_G  = re.compile(r"(\d+(?:[.,]\d+)?)\s*g(?:r|ramas?)?(?=\b|$)", re.I)
_RE_VOLTAGE   = re.compile(r"(\d+)\s*v(?:olts?)?(?=\b|$)", re.I)
_RE_BIVOLT    = re.compile(r"bi[\s-]?volt", re.I)

def _norm_str(v):
    if v is None: return None
    s = str(v).strip()
    return s if s and s.lower() not in {"null", "none", "n/a", ""} else None

def _norm_volume(v):
    if v is None: return None
    s = str(v).lower().strip()
    m = _RE_VOLUME_L.search(s)
    if m:
        n = float(m.group(1).replace(",", "."))
        return f"{int(n * 1000)}ml"
    m = _RE_VOLUME_ML.search(s)
    if m:
        n = float(m.group(1).replace(",", "."))
        return f"{int(n)}ml"
    m = _RE_WEIGHT_KG.search(s)
    if m:
        n = float(m.group(1).replace(",", "."))
        return f"{int(n) if n.is_integer() else n}kg"
    m = _RE_WEIGHT_G.search(s)
    if m:
        n = float(m.group(1).replace(",", "."))
        return f"{int(n) if n.is_integer() else n}g"
    return s.replace(" ", "")

def _norm_voltage(v):
    if v is None: return None
    s = str(v).lower().strip()
    if _RE_BIVOLT.search(s): return "bivolt"
    m = _RE_VOLTAGE.search(s)
    if m: return f"{m.group(1)}v"
    return s.replace(" ", "")

def _norm_brand(v):
    s = _norm_str(v)
    if s is None: return None
    s = re.sub(r"\bcoca[\s-]?cola\b", "Coca-Cola", s, flags=re.I)
    return " ".join(w.capitalize() if not w.isupper() or len(w) > 3 else w for w in s.split())

def _norm_lower(v):
    s = _norm_str(v)
    return s.lower() if s else None

def clean_llm_output(d):
    """Camada simbólica pós-LLM (safety net)."""
    if not isinstance(d, dict):
        return {}
    return {
        "marca":             _norm_brand(d.get("marca")),
        "modelo":            _norm_lower(d.get("modelo")),
        "tipo_produto":      _norm_lower(d.get("tipo_produto")),
        "capacidade_volume": _norm_volume(d.get("capacidade_volume")),
        "voltagem":          _norm_voltage(d.get("voltagem")),
        "cor":               _norm_lower(d.get("cor")),
        "material":          _norm_lower(d.get("material")),
        "quantidade":        _norm_lower(d.get("quantidade")),
        "outros_atributos":  _norm_lower(d.get("outros_atributos")),
    }

_KEY_ORDER = ["tipo_produto", "marca", "modelo", "capacidade_volume",
              "voltagem", "cor", "material", "quantidade", "outros_atributos"]

def build_canonical_name(cleaned):
    parts = [str(cleaned[k]) for k in _KEY_ORDER if cleaned.get(k)]
    return " | ".join(parts)

def build_match_key(cleaned):
    parts = [str(cleaned[k]).strip().lower() for k in _KEY_ORDER if cleaned.get(k)]
    return _RE_SPACES.sub(" ", "|".join(parts))

# Sanity: targets de cada grupo devem produzir o MESMO match_key (consistência por construção)
mismatched = 0
for key, group in groups.items():
    mks = set()
    for r in group:
        mks.add(build_match_key(clean_llm_output(target_dict(r))))
    if len(mks) > 1:
        mismatched += 1
print(f"Grupos com targets inconsistentes (esperado 0): {mismatched}/{len(groups)}")

# Exemplo
ex = target_dict(records[0])
print(f"\nExemplo:")
print(f"  target raw:  {ex}")
print(f"  cleaned:     {clean_llm_output(ex)}")
print(f"  canonical:   {build_canonical_name(clean_llm_output(ex))}")
print(f"  match_key:   {build_match_key(clean_llm_output(ex))}")

Grupos com targets inconsistentes (esperado 0): 0/495

Exemplo:
  target raw:  {'idx': 0, 'marca': 'Lg', 'modelo': None, 'tipo_produto': 'controle remoto + suporte ar condicionado', 'capacidade_volume': None, 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': 'original'}
  cleaned:     {'marca': 'Lg', 'modelo': None, 'tipo_produto': 'controle remoto + suporte ar condicionado', 'capacidade_volume': None, 'voltagem': None, 'cor': None, 'material': None, 'quantidade': None, 'outros_atributos': 'original'}
  canonical:   controle remoto + suporte ar condicionado | Lg | original
  match_key:   controle remoto + suporte ar condicionado|lg|original


## 6 — Gemini upper bound (sanity)

Aplica `match_key` direto nos targets dos eval records. Como os 7 records de um grupo compartilham target idêntico, a consistência é **100% por construção** — esse é o teto do benchmark.

In [ ]:
import importlib
import benchmark as bench_module
importlib.reload(bench_module)
from benchmark import run as run_benchmark

# Construir CSV no formato esperado
def records_to_benchmark_csv(records_list, normalized_fn, out_path):
    """normalized_fn(rec) → string (o match_key)."""
    rows = []
    for r in records_list:
        rows.append({
            "original":   group_to_original[target_key(r)],
            "attacked":   get_title(r),
            "normalized": normalized_fn(r),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df

# Gemini upper bound: usa o target JSON direto
def normalized_from_target(rec):
    return build_match_key(clean_llm_output(target_dict(rec)))

GEMINI_CSV     = f"{OUTPUT_DIR}/eval_gemini_upperbound.csv"
GEMINI_RESULTS = f"{OUTPUT_DIR}/results_gemini.json"

df_gemini = records_to_benchmark_csv(eval_records, normalized_from_target, GEMINI_CSV)
res_gemini = run_benchmark(Path(GEMINI_CSV), Path(GEMINI_RESULTS))

print("=" * 60)
print("  GEMINI UPPER BOUND (target → match_key direto)")
print("=" * 60)
for k, v in res_gemini["metrics"].items():
    if isinstance(v, float):
        print(f"  {k:<22s} {v:.4f}")
    else:
        print(f"  {k:<22s} {v}")
print(f"\n  Esperado: mean_constancy ~1.0 (targets idênticos por construção)")

  GEMINI UPPER BOUND (target → match_key direto)
  mean_constancy         1.0000
  min_constancy          1.0000
  max_constancy          1.0000
  num_originals          50
  total_rows             238
  weighted_constancy     1.0000

  Esperado: mean_constancy ~1.0 (targets idênticos por construção)


## 7 — Load modelo (Unsloth, 4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,        # auto: bf16 se A100, fp16 caso contrário
    load_in_4bit    = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = RANDOM_SEED,
    use_rslora     = False,
    loftq_config   = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nVRAM pós-load: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
print(f"Params treináveis: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.84k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

... (2 outputs left out)

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



VRAM pós-load: 2.42GB
Params treináveis: 29,884,416 / 2,039,024,640 (1.47%)


## 8 — Format dataset

O JSONL já está em formato `messages`. Aplicar `chat_template` do modelo direto — sem reformatar.

In [ ]:
def format_record(rec):
    """Aplica chat template e retorna {'text': '<full prompt>'}."""
    text = tokenizer.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = Dataset.from_list([format_record(r) for r in train_records])
eval_ds  = Dataset.from_list([format_record(r) for r in eval_records])

print(f"Train: {len(train_ds)}")
print(f"Eval:  {len(eval_ds)}")

# Verificar distribuição de comprimentos
lengths = []
for ex in train_ds.select(range(min(200, len(train_ds)))):
    lengths.append(len(tokenizer(ex["text"]).input_ids))
print(f"\nToken lengths (sample 200): p50={np.percentile(lengths,50):.0f}, "
      f"p95={np.percentile(lengths,95):.0f}, max={max(lengths)}")
print(f"MAX_SEQ_LENGTH config: {MAX_SEQ_LENGTH}")
if max(lengths) > MAX_SEQ_LENGTH:
    print("Algum exemplo excede MAX_SEQ_LENGTH — será truncado")

# Preview
print(records[0]["messages"][0]["content"])

Train: 2115
Eval:  238

Token lengths (sample 200): p50=1412, p95=1431, max=1440
MAX_SEQ_LENGTH config: 1536

Você é um especialista em extração de atributos de produtos de e-commerce brasileiro.

══════════════════════════════════════════════════════════════════
TAREFA
══════════════════════════════════════════════════════════════════
Você receberá um ou mais títulos de produtos, cada um identificado por um
índice numérico. Para CADA título, retorne um objeto JSON com as chaves
abaixo. Retorne um ARRAY JSON (lista) contendo um objeto por título, na
mesma ordem recebida. Se receber apenas 1 título, retorne um array com 1
objeto.

Chaves obrigatórias (use null se o atributo não existir no título):
  "idx"                : int    — o índice recebido
  "marca"              : string — ex: "Electrolux", "Samsung", "Tramontina"
  "modelo"             : string — ex: "LTD12", "Galaxy S21", null
  "tipo_produto"       : string — ex: "máquina de lavar", "geladeira"
  "capacidade_volume"  : strin

## 9 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model         = model,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = False,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 5,
        num_train_epochs            = 2,
        learning_rate               = 5e-5,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 25,
        save_strategy               = "steps",
        save_steps                  = 25,
        save_total_limit            = 3,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = RANDOM_SEED,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# Response masking — usa tokens do modelo selecionado automaticamente
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = INSTRUCTION_PART,
    response_part    = RESPONSE_PART,
)

print(f"VRAM pré-treino: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
print(f"Modelo: {MODEL_NAME}")
print(f"Instruction token: {repr(INSTRUCTION_PART)}")
print(f"Response token   : {repr(RESPONSE_PART)}")
print("Training on assistant completions ONLY")
train_stats = trainer.train()

print(f"\n── Treino concluído ──")
print(f"Loss final : {train_stats.training_loss:.4f}")
print(f"Peak VRAM  : {torch.cuda.max_memory_allocated() / 1e9:.2f}GB")


tokenizer_config.json:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2115 [00:00<?, ? examples/s]

... (9 outputs left out)

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,0.374861,0.274531
50,0.201200,0.207501
75,0.167633,0.192552
100,0.154811,0.184894
125,0.143774,0.179408
150,0.146512,0.168956
175,0.132481,0.167276
200,0.119936,0.164220
225,0.133467,0.163893
250,0.112453,0.166133


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


── Treino concluído ──
Loss final : 0.1832
Peak VRAM  : 3.66GB


## 10 — Salvar adapter

Backup no Drive recomendado pra não perder se o Colab desconectar.

In [ ]:
FINAL_ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Adapter local: {FINAL_ADAPTER_DIR}")

# Backup pro Drive
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    import shutil
    DRIVE_BACKUP = f"/content/drive/MyDrive/tail-sefaz/adapters/{RUN_NAME}"
    shutil.copytree(FINAL_ADAPTER_DIR, DRIVE_BACKUP, dirs_exist_ok=True)
    print(f"Backup Drive:  {DRIVE_BACKUP}")
except Exception as e:
    print(f"(Sem backup Drive: {e})")

Unsloth: Restored added_tokens_decoder metadata in outputs/phi-35-mini-nfe-r16/final_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/phi-35-mini-nfe-r16/final_adapter.


Adapter salvo em: outputs/phi-35-mini-nfe-r16/final_adapter
Run: phi-35-mini-nfe-r16


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado: phi-35-mini-nfe-r16_adapter.zip


## 11 — Inferência (helpers)

`for_inference` ativa kv cache e optimizations do Unsloth.

In [ ]:
FastLanguageModel.for_inference(model)

def parse_assistant_output(raw):
    """Mesmo fallback parser do pipeline v2.1."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?", "", raw).rstrip("`").strip()
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list) and parsed:
            return parsed[0]
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        m = re.search(r"\{[^{}]+\}", raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except:
                pass
    return {}

@torch.inference_mode()
def generate_batch(records_list, batch_size=8, max_new_tokens=256):
    """Gera saída pra lista de records. Retorna lista de dicts (parsed)."""
    outputs = []
    for i in range(0, len(records_list), batch_size):
        chunk = records_list[i:i + batch_size]
        # Aplicar chat template com generation prompt
        prompts = []
        for r in chunk:
            # Pegar só system + user, não assistant
            msgs = r["messages"][:2]
            prompts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            ))
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
        out = model.generate(
            **enc,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 0.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
        for j in range(len(chunk)):
            new_tokens = out[j][enc.input_ids.shape[1]:]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
            outputs.append(parse_assistant_output(raw))
    return outputs

# Smoke test em 4 títulos do eval
print("── Smoke test (4 títulos do eval) ──")
sample_recs = eval_records[:4]
sample_outs = generate_batch(sample_recs, batch_size=4)
for r, o in zip(sample_recs, sample_outs):
    print(f"\nIN:     {get_title(r)}")
    print(f"OUT:    {json.dumps(o, ensure_ascii=False)}")
    print(f"TARGET: {json.dumps(target_dict(r), ensure_ascii=False)}")

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── Smoke test (4 títulos do eval) ──


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


IN:     Balança D Precisao Digital D Cozinha Come-rcio 1g À 10kg
OUT:    {"idx": 0, "marca": null, "modelo": null, "tipo_produto": "balança", "capacidade_volume": "1g à 10kg", "voltagem": null, "cor": null, "material": null, "quantidade": null, "outros_atributos": "de precisão digital, de cozinha, com entrada"}
TARGET: {"idx": 0, "marca": null, "modelo": null, "tipo_produto": "balança", "capacidade_volume": "10kg", "voltagem": null, "cor": null, "material": null, "quantidade": null, "outros_atributos": "precisão digital, 1g"}

IN:     Balança D Precisão Digital D Cozinha Come-rcio 1g À 10kg
OUT:    {"idx": 0, "marca": null, "modelo": null, "tipo_produto": "balança", "capacidade_volume": "1g à 10kg", "voltagem": null, "cor": null, "material": null, "quantidade": null, "outros_atributos": "de precisão digital, de cozinha, com entrada"}
TARGET: {"idx": 0, "marca": null, "modelo": null, "tipo_produto": "balança", "capacidade_volume": "10kg", "voltagem": null, "cor": null, "material": null

## 12 — Rodar inferência no eval set (238 records)

In [ ]:
from tqdm.auto import tqdm

INF_BATCH = 8
print(f"Gerando outputs para {len(eval_records)} records...")
ft_outputs = []
for i in tqdm(range(0, len(eval_records), INF_BATCH), desc="Fine-tuned"):
    chunk = eval_records[i:i + INF_BATCH]
    ft_outputs.extend(generate_batch(chunk, batch_size=INF_BATCH))

assert len(ft_outputs) == len(eval_records)

# Stats: JSON validity
valid = sum(1 for o in ft_outputs if o and "tipo_produto" in o)
print(f"\nJSON valid (com tipo_produto): {valid}/{len(ft_outputs)} ({100*valid/len(ft_outputs):.1f}%)")

Gerando outputs para 238 records...


Fine-tuned:   0%|          | 0/30 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1


JSON valid (com tipo_produto): 238/238 (100.0%)


## 13 — Consistency benchmark — fine-tuned

In [ ]:
# Aplicar clean → match_key e construir CSV no formato benchmark.py
def record_to_normalized_ft(rec, idx):
    raw = ft_outputs[idx]
    cleaned = clean_llm_output(raw)
    return build_match_key(cleaned)

ft_rows = []
for i, r in enumerate(eval_records):
    ft_rows.append({
        "original":   group_to_original[target_key(r)],
        "attacked":   get_title(r),
        "normalized": record_to_normalized_ft(r, i),
    })
df_ft = pd.DataFrame(ft_rows)

FT_CSV     = f"{OUTPUT_DIR}/eval_finetuned.csv"
FT_RESULTS = f"{OUTPUT_DIR}/results_finetuned.json"
df_ft.to_csv(FT_CSV, index=False, encoding="utf-8-sig")

res_ft = run_benchmark(Path(FT_CSV), Path(FT_RESULTS))

print("=" * 60)
print(f"  CONSISTENCY BENCHMARK — FINE-TUNED ({MODEL_NAME.split('/')[-1]})")
print("=" * 60)
for k, v in res_ft["metrics"].items():
    if isinstance(v, float):
        print(f"  {k:<22s} {v:.4f}")
    else:
        print(f"  {k:<22s} {v}")

print(f"\n  Best  group: {res_ft['best_grouping']['original'][:60]}")
print(f"               constancy={res_ft['best_grouping']['constancy']:.4f}, "
      f"size={res_ft['best_grouping']['group_size']}")
print(f"  Worst group: {res_ft['worst_grouping']['original'][:60]}")
print(f"               constancy={res_ft['worst_grouping']['constancy']:.4f}, "
      f"size={res_ft['worst_grouping']['group_size']}")

# Histograma do worst group: o que o modelo produziu pras 7 variantes?
print(f"\n  Worst histogram: {res_ft['worst_grouping']['normalized_histogram']}")

  CONSISTENCY BENCHMARK — FINE-TUNED (Phi-3.5-mini-instruct)
  mean_constancy         0.3443
  min_constancy          0.1429
  max_constancy          0.7500
  num_originals          50
  total_rows             238
  weighted_constancy     0.3277

  Best  group: orig_0007
               constancy=0.7500, size=4
  Worst group: orig_0151
               constancy=0.1429, size=7

  Worst histogram: [{'normalized': 'capacete agv|valentino|gothoc white|white|k3', 'count': 1}, {'normalized': 'capacete agv|valentino|k3|white|gothic, tam 62', 'count': 1}, {'normalized': 'capacete|branco|agv k3 gothic, valentino, tam 62', 'count': 1}, {'normalized': 'capacete|valentino|k3 gothuc|white|tam 62|agv', 'count': 1}, {'normalized': 'capacete|valentino|k3|white|agv, gothic, 62', 'count': 1}, {'normalized': 'capacete|valentino|k3|white|agv, gothic, tam. 62', 'count': 1}, {'normalized': 'capacete|valentino|k3|white|tam 62|agv, gothic', 'count': 1}]


## 14 — Field-level agreement com Gemini

Constância sozinha é gameável (modelo que sempre cospe `{}` tem 100% constância). Verifica se o modelo está sendo consistente pelo **motivo certo** — i.e. seus campos batem com os do target.

In [ ]:
def field_agreement(predictions, targets, fields=None):
    """% de campos onde pred == target (após cleaning), por campo e total."""
    if fields is None:
        fields = _KEY_ORDER
    counts = {f: [0, 0] for f in fields}  # [matches, total]
    for pred_raw, target_raw in zip(predictions, targets):
        p = clean_llm_output(pred_raw)
        t = clean_llm_output(target_raw)
        for f in fields:
            counts[f][1] += 1
            if p.get(f) == t.get(f):
                counts[f][0] += 1
    return {f: counts[f][0] / counts[f][1] for f in fields}

ft_field_agree = field_agreement(
    ft_outputs,
    [target_dict(r) for r in eval_records],
)

print("Field-level agreement (fine-tuned vs Gemini):")
for f, score in sorted(ft_field_agree.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 20)
    print(f"  {f:<22s} {score:.4f} {bar}")
mean_agree = sum(ft_field_agree.values()) / len(ft_field_agree)
print(f"\n  Mean field agreement: {mean_agree:.4f}")

Field-level agreement (fine-tuned vs Gemini):
  voltagem               1.0000 ████████████████████
  capacidade_volume      0.9706 ███████████████████
  material               0.8739 █████████████████
  cor                    0.8655 █████████████████
  quantidade             0.7899 ███████████████
  modelo                 0.7143 ██████████████
  marca                  0.7059 ██████████████
  tipo_produto           0.4496 ████████
  outros_atributos       0.0420 

  Mean field agreement: 0.7124


## 15 — Zero-shot baseline (mesmo modelo, sem treino)

Carrega o base model puro pra ver quanto o fine-tuning agregou. Usa exatamente o mesmo system prompt.

In [ ]:
# Limpa fine-tuned
del model, trainer
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,
    load_in_4bit    = LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(base_model)

@torch.inference_mode()
def generate_batch_base(records_list, batch_size=8, max_new_tokens=256):
    outputs = []
    for i in range(0, len(records_list), batch_size):
        chunk = records_list[i:i + batch_size]
        prompts = [base_tokenizer.apply_chat_template(
            r["messages"][:2], tokenize=False, add_generation_prompt=True
        ) for r in chunk]
        enc = base_tokenizer(prompts, return_tensors="pt", padding=True,
                              truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
        out = base_model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            temperature=0.0, pad_token_id=base_tokenizer.eos_token_id,
        )
        for j in range(len(chunk)):
            new_tokens = out[j][enc.input_ids.shape[1]:]
            raw = base_tokenizer.decode(new_tokens, skip_special_tokens=True)
            outputs.append(parse_assistant_output(raw))
    return outputs

print(f"Rodando zero-shot em {len(eval_records)} records...")
zs_outputs = []
for i in tqdm(range(0, len(eval_records), INF_BATCH), desc="Zero-shot"):
    chunk = eval_records[i:i + INF_BATCH]
    zs_outputs.extend(generate_batch_base(chunk, batch_size=INF_BATCH))

zs_valid = sum(1 for o in zs_outputs if o and "tipo_produto" in o)
print(f"\nJSON valid: {zs_valid}/{len(zs_outputs)} ({100*zs_valid/len(zs_outputs):.1f}%)")

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Rodando zero-shot em 238 records...


Zero-shot:   0%|          | 0/30 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


JSON valid: 224/238 (94.1%)


## 16 — Benchmark zero-shot

In [ ]:
zs_rows = []
for i, r in enumerate(eval_records):
    cleaned = clean_llm_output(zs_outputs[i])
    zs_rows.append({
        "original":   group_to_original[target_key(r)],
        "attacked":   get_title(r),
        "normalized": build_match_key(cleaned),
    })
df_zs = pd.DataFrame(zs_rows)

ZS_CSV     = f"{OUTPUT_DIR}/eval_zeroshot.csv"
ZS_RESULTS = f"{OUTPUT_DIR}/results_zeroshot.json"
df_zs.to_csv(ZS_CSV, index=False, encoding="utf-8-sig")

res_zs = run_benchmark(Path(ZS_CSV), Path(ZS_RESULTS))

print("=" * 60)
print(f"  CONSISTENCY BENCHMARK — ZERO-SHOT ({MODEL_NAME.split('/')[-1]})")
print("=" * 60)
for k, v in res_zs["metrics"].items():
    if isinstance(v, float):
        print(f"  {k:<22s} {v:.4f}")
    else:
        print(f"  {k:<22s} {v}")

zs_field_agree = field_agreement(
    zs_outputs,
    [target_dict(r) for r in eval_records],
)
print("\nField-level agreement (zero-shot vs Gemini):")
for f, score in sorted(zs_field_agree.items(), key=lambda x: -x[1]):
    print(f"  {f:<22s} {score:.4f}")
print(f"\n  Mean field agreement: {sum(zs_field_agree.values())/len(zs_field_agree):.4f}")

  CONSISTENCY BENCHMARK — ZERO-SHOT (Phi-3.5-mini-instruct)
  mean_constancy         0.3177
  min_constancy          0.1429
  max_constancy          1.0000
  num_originals          50
  total_rows             238
  weighted_constancy     0.2899

Field-level agreement (zero-shot vs Gemini):
  cor                    0.8739
  material               0.8697
  voltagem               0.8613
  capacidade_volume      0.8109
  quantidade             0.6933
  marca                  0.4748
  modelo                 0.3739
  tipo_produto           0.2647
  outros_atributos       0.0294

  Mean field agreement: 0.5836


## 17 — Tabela final comparativa

In [ ]:
def fmt_pct(v):
    return f"{v*100:.2f}%" if v is not None else "  —  "

print()
print("┌─────────────────────────────────────────────────────────────────────────────┐")
print(f"│  RESULTADOS — Holdout: {N_EVAL_ORIGS} originais × 7 variantes = {len(eval_records)} records           │")
print("├─────────────────────────────┬──────────────┬──────────────┬──────────────────┤")
print("│  Sistema                    │ mean_const.  │ weighted_c.  │ field_agreement  │")
print("├─────────────────────────────┼──────────────┼──────────────┼──────────────────┤")

systems = [
    ("Gemini (upper bound)",  res_gemini, None),
    (f"Qwen-{MODEL_NAME.split('-')[-2]} zero-shot",       res_zs,     zs_field_agree),
    (f"Qwen-{MODEL_NAME.split('-')[-2]} fine-tuned",      res_ft,     ft_field_agree),
]
for name, res, fa in systems:
    mc = res["metrics"]["mean_constancy"]
    wc = res["metrics"]["weighted_constancy"]
    fa_mean = sum(fa.values()) / len(fa) if fa else 1.0
    print(f"│  {name:<27s} │  {fmt_pct(mc):<10s}  │  {fmt_pct(wc):<10s}  │  {fmt_pct(fa_mean):<14s}  │")
print("└─────────────────────────────┴──────────────┴──────────────┴──────────────────┘")

# Delta fine-tuning
delta_mc = res_ft["metrics"]["mean_constancy"] - res_zs["metrics"]["mean_constancy"]
delta_fa = (sum(ft_field_agree.values()) - sum(zs_field_agree.values())) / len(ft_field_agree)
print(f"\n  Δ mean_constancy   (FT − ZS):  {delta_mc:+.4f}")
print(f"  Δ field_agreement  (FT − ZS):  {delta_fa:+.4f}")

# Cost estimate (estimativa grosseira)
print(f"\n  Custo de inferência (10M itens, comparativo):")
print(f"    Gemini Flash-Lite:   ~US$ 1.831,50  (do model_v2_1.ipynb)")
print(f"    Qwen-3B local L4:    ~US$ 25-40  (50 it/s, US$0.50/h Colab)")
print(f"    Trade-off real:      data residency (LGPD) + sem rate limit + custo previsível")


┌─────────────────────────────────────────────────────────────────────────────┐
│  RESULTADOS — Holdout: 50 originais × 7 variantes = 238 records           │
├─────────────────────────────┬──────────────┬──────────────┬──────────────────┤
│  Sistema                    │ mean_const.  │ weighted_c.  │ field_agreement  │
├─────────────────────────────┼──────────────┼──────────────┼──────────────────┤
│  Gemini (upper bound)        │  100.00%     │  100.00%     │  100.00%         │
│  Qwen-mini zero-shot         │  31.77%      │  28.99%      │  58.36%          │
│  Qwen-mini fine-tuned        │  34.43%      │  32.77%      │  71.24%          │
└─────────────────────────────┴──────────────┴──────────────┴──────────────────┘

  Δ mean_constancy   (FT − ZS):  +0.0266
  Δ field_agreement  (FT − ZS):  +0.1289

  Custo de inferência (10M itens, comparativo):
    Gemini Flash-Lite:   ~US$ 1.831,50  (do model_v2_1.ipynb)
    Qwen-3B local L4:    ~US$ 25-40  (50 it/s, US$0.50/h Colab)
    Trade-off

## 18 — Salvar artefatos pro Drive

In [ ]:
try:
    import shutil
    DRIVE_OUT = f"/content/drive/MyDrive/tail-sefaz/results/{RUN_NAME}"
    shutil.copytree(OUTPUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
    print(f"✓ Outputs copiados pra: {DRIVE_OUT}")
    print(f"  Arquivos:")
    for p in sorted(Path(OUTPUT_DIR).glob("**/*")):
        if p.is_file():
            print(f"    - {p.relative_to(OUTPUT_DIR)}")
except Exception as e:
    print(f"(Sem backup Drive: {e})")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download: outputs/phi-35-mini-nfe-r16/eval_finetuned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download: outputs/phi-35-mini-nfe-r16/eval_zeroshot.csv


... (2 outputs left out)

Download: outputs/phi-35-mini-nfe-r16/results_finetuned.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download: outputs/phi-35-mini-nfe-r16/results_zeroshot.json
